In [3]:
import os
from math import nan
import pandas as pd
pd.options.plotting.backend='plotly'

# Grid Search Results

```
Model                   SkillDaily  SkillsPos
u24-96_d0.1_n48_f4      0.435722    0.611111
u48-24_d0.1_n24_f4      0.415693    0.594444
u128-96_d0.1_n24_f4     0.413271    0.605556
```

In [8]:
#results_dir = r'results/redcliff_healthcenter/gridsearch v1.7 250324'
results_dir = r'results/jpl_ev/hpsearch_v1-8_250326/'

results = pd.DataFrame({'Model':[],'Skill':[],'Daily Skills Pos':[]})

for model in [x for x in os.listdir(results_dir) if x[0] == 'u']:
    model_dir = results_dir + r'/' + model
    
    if 'all_forecasts.csv' in os.listdir(model_dir):

        df = pd.read_csv(model_dir + r'/all_forecasts.csv',index_col=0)

        df['ErrorPred'] = (df.Pred - df.Load)/df.Load.max()
        df['ErrorPers'] = (df.Persist - df.Load)/df.Load.max()

        mae_pers = df[df.ErrorPers.abs()>0.0].ErrorPred.abs().mean()
        mae_pred = df[df.ErrorPers.abs()>0.0].ErrorPers.abs().mean()
        
        grouped = df.groupby('timestamp_update')
        skills = []
        for name, group in grouped:
            mae_pers = group['ErrorPers'].abs().mean()
            mae_pred = group['ErrorPred'].abs().mean()
            if mae_pers > 0:
                skill = 1 - mae_pred / mae_pers
            else:
                skill = nan
            skills.append(skill)
        daily_skills_pos = len([x for x in skills if x > 0])/len(skills)
        
    else:
        mae_pers = nan
        mae_pred = nan
        daily_skills_pos = nan
    
    results.loc[len(results)] = {'Model':model,'Skill':1 - mae_pred / mae_pers,'Daily Skills Pos':daily_skills_pos}
        
results.sort_values('Skill',ascending=False)

,Model,Skill,Daily Skills Pos
182,u24-96_d0.1_n48_f4,0.435722,0.611111
311,u48-24_d0.1_n24_f4,0.415693,0.594444
112,u128-96_d0.1_n24_f4,0.413271,0.605556
191,u256-128_d0_n24_f1,0.398106,0.588889
401,u8-96_d0.1_n48_f3,0.397099,0.591667
...,...,...,...
461,u96-96_d0.1_n384_f3,NaN,NaN
464,u96-96_d0_n192_f1,NaN,NaN
465,u96-96_d0_n192_f3,NaN,NaN
466,u96-96_d0_n192_f4,NaN,NaN
